This notebook deals with the part 2 of the project.
- I did the initial EDA , Data cleaning and Data Imputation of the Median_Income data set 
- Saved the cleaned median data as a parquet file

In [2]:
import pandas as pd ,numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_timestamp, hour, month, year, when, count, desc, dayofmonth, to_date

# Median Income Data Set:

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

# Initialize Spark with the Postgres driver
spark = SparkSession.builder \
    .appName("LA-Crime-Research") \
    .getOrCreate()
    
# Load the Median Income Data 
df_income = spark.read.csv("CS226_project/Median_Income_and_AMI_(census_tract).csv", header=True, inferSchema=True)
df_income.show(5)


ERROR StatusLogger Could not determine local host name
 java.net.UnknownHostException: e40014039e74: e40014039e74: Temporary failure in name resolution
	at java.base/java.net.InetAddress.getLocalHost(InetAddress.java:1936)
	at org.apache.logging.log4j.core.util.NetUtils.getLocalHostname(NetUtils.java:56)
	at org.apache.logging.log4j.core.LoggerContext.lambda$setConfiguration$0(LoggerContext.java:615)
	at java.base/java.util.concurrent.ConcurrentHashMap.computeIfAbsent(ConcurrentHashMap.java:1708)
	at org.apache.logging.log4j.core.LoggerContext.setConfiguration(LoggerContext.java:615)
	at org.apache.logging.log4j.core.LoggerContext.reconfigure(LoggerContext.java:694)
	at org.apache.logging.log4j.core.LoggerContext.reconfigure(LoggerContext.java:711)
	at org.apache.logging.log4j.core.LoggerContext.start(LoggerContext.java:253)
	at org.apache.logging.log4j.core.impl.Log4jContextFactory.getContext(Log4jContextFactory.java:245)
	at org.apache.logging.log4j.core.impl.Log4jContextFactory.getC

ERROR StatusConsoleListener Could not determine local host name
 java.net.UnknownHostException: e40014039e74: e40014039e74: Temporary failure in name resolution
	at java.base/java.net.InetAddress.getLocalHost(InetAddress.java:1936)
	at org.apache.logging.log4j.core.util.NetUtils.getLocalHostname(NetUtils.java:56)
	at org.apache.logging.log4j.core.LoggerContext.lambda$setConfiguration$0(LoggerContext.java:615)
	at java.base/java.util.concurrent.ConcurrentHashMap.computeIfAbsent(ConcurrentHashMap.java:1708)
	at org.apache.logging.log4j.core.LoggerContext.setConfiguration(LoggerContext.java:615)
	at org.apache.logging.log4j.core.LoggerContext.reconfigure(LoggerContext.java:694)
	at org.apache.logging.log4j.core.LoggerContext.setConfigLocation(LoggerContext.java:679)
	at org.apache.spark.internal.Logging.initializeLogging(Logging.scala:139)
	at org.apache.spark.internal.Logging.initializeLogIfNecessary(Logging.scala:114)
	at org.apache.spark.internal.Logging.initializeLogIfNecessary$(Loggi

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/04 18:07:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/04 18:07:07 WARN MacAddressUtil: Failed to find a usable hardware address from the network interfaces; using random bytes: 1a:33:0b:6b:be:5d:46:4a


+----------+-------------+----------------------+---------------+----------------+----------------------+---------------------+----------+--------------------+--------------------+--------+------------------+----------------+
|     tract|med_hh_income|med_hh_income_universe|   ami_category|below_med_income|below_60pct_med_income|below_moderate_income|  sup_dist|                 csa|                 spa|ESRI_OID|       Shape__Area|   Shape__Length|
+----------+-------------+----------------------+---------------+----------------+----------------------+---------------------+----------+--------------------+--------------------+--------+------------------+----------------+
|6037101110|        84091|                  1558|     Low Income|             Yes|                    No|                  Yes|District 5|Los Angeles - Tuj...|SPA 2 - San Fernando|    4842|1.23298062148438E7|14765.6490038758|
|6037101122|        99583|                  1407|     Low Income|              No|              

In [5]:
df_income.printSchema()

root
 |-- tract: string (nullable = true)
 |-- med_hh_income: string (nullable = true)
 |-- med_hh_income_universe: string (nullable = true)
 |-- ami_category: string (nullable = true)
 |-- below_med_income: string (nullable = true)
 |-- below_60pct_med_income: string (nullable = true)
 |-- below_moderate_income: string (nullable = true)
 |-- sup_dist: string (nullable = true)
 |-- csa: string (nullable = true)
 |-- spa: string (nullable = true)
 |-- ESRI_OID: string (nullable = true)
 |-- Shape__Area: string (nullable = true)
 |-- Shape__Length: string (nullable = true)



## 2. EDA + Data Cleaning:


Used Pandas for this purpose

#### 1. Overview:

In [4]:

data = pd.read_csv("CS226_project/Median_Income_and_AMI_(census_tract).csv")
data.head(5)

# 1. Data set Overview:
print("Dataset Overview")
print(f"Total Rows: {data.shape[0]}, Total Columns: {data.shape[1]}")

# 2. Calculate the number of nulls and the percentage:
print("% of Null values per column")
null_counts = data.isnull().sum()
null_percentages = (null_counts / len(data)) * 100
# Create a summary table
null_summary = pd.DataFrame({
    'Null Count': null_counts,
    'Percentage (%)': null_percentages
})
print(null_summary)
      

Dataset Overview
Total Rows: 2495, Total Columns: 13
% of Null values per column
                        Null Count  Percentage (%)
tract                            0        0.000000
med_hh_income                   42        1.683367
med_hh_income_universe           0        0.000000
ami_category                    42        1.683367
below_med_income                42        1.683367
below_60pct_med_income          42        1.683367
below_moderate_income           42        1.683367
sup_dist                         0        0.000000
csa                              0        0.000000
spa                              0        0.000000
ESRI_OID                         0        0.000000
Shape__Area                      0        0.000000
Shape__Length                    0        0.000000


**Observation**

In this dataset, exactly 42 census tracts (1.68%) are missing income-specific data.
- Geographic/ID (tract, spa, ESRI_OID)  00.00%
- Income Metrics (med_hh_income, ami_category)   1.68%

#### 2. Socio Economic Insights:

In [5]:
# Income stats by region:
avg_income_spa = data.groupby('spa')['med_hh_income'].mean().sort_values(ascending=False)
print("\n--- Average Income by Service Planning Area (SPA) ---")
print(avg_income_spa)

# Distribution of AMI categories:
print("\n--- AMI Category Distribution ---")
print(data['ami_category'].value_counts())


--- Average Income by Service Planning Area (SPA) ---
spa
SPA 5 - West               131263.522222
SPA 2 - San Fernando       102003.381387
SPA 8 - South Bay          100073.457534
SPA 3 - San Gabriel         99429.498721
SPA 7 - East                86813.947735
SPA 1 - Antelope Valley     83880.122222
SPA 4 - Metro               77333.547550
SPA 6 - South               61994.008163
Name: med_hh_income, dtype: float64

--- AMI Category Distribution ---
ami_category
Low Income               1094
Very Low Income           638
Above Moderate Income     527
Moderate Income           104
Extremely Low Income       85
Acutely Low Income          5
Name: count, dtype: int64


**Observations**

- **Integrity**: Approximately 1.68% of census tracts (42 out of 2,495) are missing income data and were removed during cleaning.

- **Wealth Gap**: The West (SPA 5) is the wealthiest region with an average income of ~$131,263, while the South (SPA 6) is the lowest at ~$61,994.

- **Dominant Group**: "Low Income" is the most frequent tract classification in Los Angeles.

#### 3. Data Cleaning:

In [6]:
# 1. Deal with missing values:

# Calculate mean income for the 'Low Income' group
low_income_mean = data[data['ami_category'] == 'Low Income']['med_hh_income'].mean()
print(f"Mean income for 'Low Income' group: {low_income_mean}")

# Perform the replacement
data_imputed = data.copy()
data_imputed['med_hh_income'] = data_imputed['med_hh_income'].fillna(low_income_mean)

# Verify nulls are gone in med_hh_income
new_nulls = data_imputed['med_hh_income'].isnull().sum()
print(f"New null count for med_hh_income: {new_nulls}")

# Show a few rows that were previously null (if possible)
# Finding indices of original nulls
null_indices = data[data['med_hh_income'].isnull()].index
print("\nSample of imputed values:")
print(data_imputed.loc[null_indices, ['tract', 'med_hh_income']].head())

Mean income for 'Low Income' group: 88984.63345521023
New null count for med_hh_income: 0

Sample of imputed values:
          tract  med_hh_income
36   6037115103   88984.633455
461  6037190303   88984.633455
569  6037204410   88984.633455
598  6037207307   88984.633455
938  6037265301   88984.633455


In [7]:
# 2. Standardize 'tract' as a string to prevent joining errors
data_imputed['tract'] = data_imputed['tract'].astype(str)

# 3. Standardize text data
data_imputed['spa'] = data_imputed['spa'].str.strip()


## 3. Save the cleaned File :

In [8]:
# Save cleaned local copy
data_imputed.to_csv('Median_Income_Cleaned.csv', index=False)
print("\nCleaned local file saved as: Median_Income_Cleaned.csv")


Cleaned local file saved as: Median_Income_Cleaned.csv
